# Notebook 1 — Build the Network
**YTS+ DSEB 2026 · Project B · Indian Railways**

---

Indian Railways runs over 13,000 trains a day across more than 8,000 stations.
Every train has a public schedule — a list of stations it stops at, with times.

Someone collected every schedule and put it online. We downloaded it.
By the end of this session you will have built a map of every physical railway track in India — using nothing but train timetables.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs('images', exist_ok=True)
DATA = 'data/'
print('Ready.')

---
## Part 1 — What does the raw data look like?

The original file is `schedules.json` — **417,080 rows**. Each row is one station stop for one train.

Here are 6 rows from it. This is train 12301 — the Howrah Rajdhani Express, New Delhi → Kolkata:

| train_number | station_code | departure | day | sequence |
|---|---|---|---|---|
| 12301 | NDLS (New Delhi) | 16:55 | 1 | 1 |
| 12301 | CNB (Kanpur) | 22:40 | 1 | 2 |
| 12301 | ALD (Prayagraj) | 01:15 | 2 | 3 |
| 12301 | MGS (Mughal Sarai) | 03:10 | 2 | 4 |
| 12301 | PNBE (Patna) | 07:00 | 2 | 5 |
| 12301 | HWH (Howrah) | 10:30 | 2 | 6 |

**One row = one train halting at one station.** That is the entire raw material.

From this we can extract: for each train, which two stations appear as consecutive stops? That is the connection between them.

### The stations

In [ ]:
# Each row is one station. Columns: code, name, state, lat (latitude), lon (longitude)
stations = pd.read_csv(DATA + 'stations_metrics.csv')

print(f'Total stations: {len(stations):,}')
print()
stations[['code','name','state','lat','lon']].head(10)

In [ ]:
# Which states have the most stations?
by_state = stations[stations['state'] != ''].groupby('state').size()
by_state = by_state.sort_values(ascending=False)

print('Stations per state (top 15):')
print(by_state.head(15).to_string())

In [ ]:
# Plot every station as a dot at its GPS location
fig, ax = plt.subplots(figsize=(8, 10))
ax.scatter(stations['lon'], stations['lat'], s=0.4, color='steelblue', alpha=0.4)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'India — all {len(stations):,} railway stations')
ax.set_xlim(68, 97)
ax.set_ylim(8, 37)
plt.tight_layout()
plt.savefig('images/all_stations.png', dpi=150)
plt.show()

**Write:** Where are stations dense? Where are they sparse? Name one region that seems to have very few stations and suggest why.

*Your observation:*

### The raw connections between stations

We processed all 417,080 rows and extracted every pair of **consecutive stops** for every train. If train 12301 stops at Kanpur then Prayagraj, that is one pair.

The result is `edges.csv` — every consecutive stop pair with how many trains share it.

> **Important:** this is raw data. It includes both real track connections and "skip edges" — cases where an express train jumped over stations without stopping. We will fix that in Part 3.

Each edge has a `num_trains` column — how many different trains run between those two stations. This is the **edge weight**. It measures train volume, not passengers — one Rajdhani Express and one slow local passenger train both count as 1.

In [ ]:
# Load the raw consecutive stop pairs
edges_raw = pd.read_csv(DATA + 'edges.csv')

print(f'Total consecutive stop pairs (raw, includes skip edges): {len(edges_raw):,}')
print()
edges_raw.head(8)

### The trains

In [ ]:
trains = pd.read_csv(DATA + 'train_stats.csv')

print(f'Total trains: {len(trains):,}')
print()
print('What each column means:')
print('  n_stops           — how many stations this train halts at')
print('  total_distance_km — total route length (straight-line km between stops)')
print('  avg_km_per_stop   — average km between each halt')
print('  n_skip_segments   — how many times the train jumps over a station without stopping')
print()
trains.head(8)

---
## Part 2 — Express trains vs slow trains

`avg_km_per_stop` tells you how express a train is.

- A slow local train stopping every 5 km → `avg_km_per_stop = 5`  
- A Rajdhani stopping every 200 km → `avg_km_per_stop = 200`

**Before you compute:** Do you think trains that travel longer distances skip more stations? Does a Mumbai–Delhi train stop less often than a Delhi–Agra train?

*Your prediction (write before running the next cell):*

In [ ]:
print('Average km between consecutive halts — all trains:')
print(trains['avg_km_per_stop'].describe().round(1))
print()
print(f"Trains stopping every 10 km or less: {(trains['avg_km_per_stop'] <= 10).sum():,}")
print(f"Trains stopping every 30 km or more: {(trains['avg_km_per_stop'] >= 30).sum():,}")
print(f"Trains with ANY station skipped:     {(trains['n_skip_segments'] > 0).sum():,}")

In [ ]:
# Does a longer journey mean more skipping?
# Split trains into distance buckets and compare

short  = trains[trains['total_distance_km'] <  500]
medium = trains[(trains['total_distance_km'] >= 500) & (trains['total_distance_km'] < 1500)]
long   = trains[trains['total_distance_km'] >= 1500]

print(f"Short trains  (< 500 km):   {len(short):,} trains,  median {short['avg_km_per_stop'].median():.1f} km/stop")
print(f"Medium trains (500–1500 km): {len(medium):,} trains,  median {medium['avg_km_per_stop'].median():.1f} km/stop")
print(f"Long trains   (> 1500 km):  {len(long):,} trains,  median {long['avg_km_per_stop'].median():.1f} km/stop")
print()
print('Notice: the median is almost the same across all three groups.')
print('In India, even very long trains stop nearly as often as short ones.')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(trains['total_distance_km'], trains['avg_km_per_stop'],
           s=3, alpha=0.2, color='steelblue')

# Highlight the genuine express outliers
express = trains[(trains['total_distance_km'] > 800) & (trains['avg_km_per_stop'] > 50)]
ax.scatter(express['total_distance_km'], express['avg_km_per_stop'],
           s=30, color='tomato', zorder=5, label=f'Genuine express ({len(express)} trains)')

ax.set_xlabel('Total route distance (km)')
ax.set_ylabel('Average km between consecutive halts')
ax.set_title('Most Indian trains cluster below 15 km/stop regardless of distance')
ax.legend()
plt.tight_layout()
plt.savefig('images/express_scatter.png', dpi=150)
plt.show()

**Write:** Were you right? What does it mean that even a 2000 km train stops every 7–8 km on average? What would that feel like as a passenger? What does it tell you about who Indian Railways serves?

*Your answer:*

---
## Part 3 — How do we build the physical track from timetables?

We connected every pair of consecutive stops. That gives us a lot of connections — but some are fake.

Look at the Howrah Rajdhani schedule again:

> New Delhi → **Kanpur** → Prayagraj → Mughal Sarai → Patna → Howrah

When we connect consecutive stops, we get the edge **New Delhi → Kanpur** (440 km).
That is a real track.

But another express train might go directly:
> New Delhi → Patna (skipping Kanpur and Prayagraj)

Now we have the edge **New Delhi → Patna** (1,000 km). That looks like a direct track — but it is not. The train simply did not stop in between.

**These fake long connections are called skip edges.** We need to remove them.

### Why we need distances: the haversine formula

To test whether edge A→C is a skip, we need to measure how far apart stations are. But the data gives us **latitude and longitude** — angles, not distances.

You cannot simply subtract coordinates. Here is why:
- 1° of latitude = roughly 111 km everywhere in India ✓
- 1° of longitude = 111 km at the equator, about 98 km at Delhi (28°N), and shrinks toward zero at the poles ✗

And the Earth is a sphere. The straight-line distance between Delhi and Mumbai passes through the Earth. What we want is the **surface distance** — the arc along the curved surface.

The **haversine formula** takes two GPS points (lat, lon) and returns the surface distance in kilometres.

**Before you run the next cell:** Delhi is at (28.6°N, 77.2°E). Mumbai is at (19.1°N, 72.9°E). What do you expect the surface distance to be?

*Your estimate:*

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    """Surface distance between two GPS points, in km."""
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

# Verify with known pairs
print(f'Delhi  to Mumbai:  {haversine(28.6, 77.2, 19.1, 72.9):.0f} km  (expect ~1,150)')
print(f'Delhi  to Chennai: {haversine(28.6, 77.2, 13.1, 80.3):.0f} km  (expect ~2,200)')
print(f'Mumbai to Kolkata: {haversine(19.1, 72.9, 22.6, 88.4):.0f} km  (expect ~1,650)')
print()
print('Check these on Google Maps. If they are roughly right, the formula works.')

In [ ]:
### Step 2 — The common neighbor test

Distance alone cannot identify skip edges. There are genuine physical track connections over 800 km in remote desert and mountain areas of India.

We need a smarter test. The key insight:

> **If station B is already connected to both A and C by trains, AND B lies geographically between A and C, then the train running A→C directly must have passed through B without stopping.**

Station B is a **common neighbor** of A and C if trains stop at A→B AND at C→B. B is in the graph, connected to both endpoints.

The geometry check: `distance(A, B) + distance(B, C) < 1.3 × distance(A, C)`

If this holds, B sits between A and C — the 1.3 factor allows for natural track curvature, since physical rails are never perfectly straight.

**Why only common neighbors?** We do not check all 8,697 stations — only those already connected to both A and C. This is fast because the number of common neighbors per edge is small.

**The algorithm:**
1. For each edge where distance > 60 km
2. Find all common neighbors of its two endpoints
3. If any common neighbor B passes the geometry test → mark A→C as a skip edge
4. Remove all skip edges → physical track network

In [ ]:
import networkx as nx
import time

# ── Step 1: Add haversine distances to every raw edge ─────────────────────────
coord_dict = stations.set_index('code')[['lat', 'lon']].to_dict('index')

def edge_dist(row):
    a, b = row['station_a'], row['station_b']
    if a in coord_dict and b in coord_dict:
        return haversine(coord_dict[a]['lat'], coord_dict[a]['lon'],
                         coord_dict[b]['lat'], coord_dict[b]['lon'])
    return None

edges_raw['distance_km'] = edges_raw.apply(edge_dist, axis=1)

print(f'Distances added to all {len(edges_raw):,} edges')
print(f'Edges longer than 60 km (suspicious): {(edges_raw["distance_km"] > 60).sum()}')

# Distance distribution
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(edges_raw['distance_km'].clip(upper=150), bins=80, color='steelblue', edgecolor='white')
ax.axvline(60, color='tomato', linewidth=1.5, linestyle='--', label='60 km threshold')
ax.set_xlabel('Distance between consecutive stops (km)')
ax.set_ylabel('Number of edges')
ax.set_title('Most track connections are very short — a small tail of long edges exists')
ax.legend()
plt.tight_layout()
plt.savefig('images/edge_distance_dist.png', dpi=150)
plt.show()

# ── Step 2: Build the stop network ───────────────────────────────────────────
G_stop = nx.Graph()
for _, row in edges_raw.iterrows():
    a, b = row['station_a'], row['station_b']
    if a in coord_dict and b in coord_dict:
        G_stop.add_edge(a, b, weight=int(row['num_trains']))

print(f'\nStop network: {G_stop.number_of_nodes():,} stations, {G_stop.number_of_edges():,} edges')

# ── Step 3: Common neighbor test ─────────────────────────────────────────────
MIN_KM        = 60
DETOUR_FACTOR = 1.3

t0 = time.time()
skip_edges = []

for a, c in list(G_stop.edges()):
    if a not in coord_dict or c not in coord_dict:
        continue
    lat_a, lon_a = coord_dict[a]['lat'], coord_dict[a]['lon']
    lat_c, lon_c = coord_dict[c]['lat'], coord_dict[c]['lon']
    dist_ac = haversine(lat_a, lon_a, lat_c, lon_c)

    if dist_ac < MIN_KM:
        continue  # short — definitely real track

    # Only check common neighbors of A and C
    common_nbrs = set(G_stop.neighbors(a)) & set(G_stop.neighbors(c))
    for b in common_nbrs:
        if b not in coord_dict:
            continue
        lat_b, lon_b = coord_dict[b]['lat'], coord_dict[b]['lon']
        dist_ab = haversine(lat_a, lon_a, lat_b, lon_b)
        dist_bc = haversine(lat_b, lon_b, lat_c, lon_c)
        if dist_ab + dist_bc < DETOUR_FACTOR * dist_ac:
            skip_edges.append((a, c))
            break

print(f'Skip edges found: {len(skip_edges)} in {time.time()-t0:.1f}s')

# ── Step 4: Remove skip edges → physical track network ───────────────────────
G_track = G_stop.copy()
G_track.remove_edges_from(skip_edges)
print(f'Physical track: {G_track.number_of_nodes():,} stations, {G_track.number_of_edges():,} edges')

# Verify against the pre-built reference
track_ref = pd.read_csv(DATA + 'track_edges.csv')
match = G_track.number_of_edges() == len(track_ref)
print(f'Match with reference: {"✓  Exact match" if match else "✗ — check MIN_KM and DETOUR_FACTOR"}')

# ── Before / after map ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8))

for ax, G, title in [
    (ax1, G_stop,  f'BEFORE — {G_stop.number_of_edges():,} edges (includes express jumps)'),
    (ax2, G_track, f'AFTER  — {G_track.number_of_edges():,} edges (physical track only)'),
]:
    for a, b in G.edges():
        if a in coord_dict and b in coord_dict:
            ax.plot([coord_dict[a]['lon'], coord_dict[b]['lon']],
                    [coord_dict[a]['lat'], coord_dict[b]['lat']],
                    color='steelblue', linewidth=0.2, alpha=0.4)
    ax.set_xlim(68, 97)
    ax.set_ylim(8, 37)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.suptitle('Removing skip edges recovers the physical railway map of India', fontsize=12)
plt.tight_layout()
plt.savefig('images/before_after_track.png', dpi=150)
plt.show()

# ── Load enriched track for Part 4 ───────────────────────────────────────────
# Same network you just built — with station names, states, and distances pre-joined
track = pd.read_csv(DATA + 'track_edges_enriched.csv')
print(f'\nEnriched track loaded for Part 4: {len(track):,} connections')
print(f'Columns: {track.columns.tolist()}')

**Write:** Why do you think so few skip edges exist? What does it tell you about how Indian Railways was designed — who is it built to serve?

*Your answer:*

---
## Part 4 — Five questions about Indian Railways

We have the full physical track network. Let us use it to answer five real questions.

### Question 1 — Which train makes the biggest single jump between consecutive stops?

Even after removing skip edges, some physical track stretches are very long — places where there genuinely is no station in between.

In [ ]:
# Sort by distance to find the longest physical track connections
longest = track.sort_values('distance_km', ascending=False)

print('10 longest physical track connections (no station in between):')
print()
print(f'{"From":<28} {"To":<28} {"Distance":>10} {"Trains":>8}')
print('-' * 76)
for _, row in longest.head(10).iterrows():
    print(f"{str(row['name_a']):<28} {str(row['name_b']):<28} {row['distance_km']:>8.0f} km {int(row['num_trains']):>7}")

**Write:** Look these places up on a map. What geographic features (sea, desert, mountains, border) explain why there is no station between them?

*Your answer:*

### Question 2 — How many stations are dead ends?

A dead end station has only one track connection. If that connection breaks — flood, landslide, broken bridge — the station has no way in or out.

We count connections per station by counting how many times each station appears in the track data.

In [ ]:
# Count how many track connections each station has
from_a = track.groupby('station_a').size()
from_b = track.groupby('station_b').size()
connections = from_a.add(from_b, fill_value=0).astype(int)

dead_ends    = connections[connections == 1]
pass_through = connections[connections == 2]
junctions    = connections[connections >= 3]

print(f'Dead end stations     (1 connection):   {len(dead_ends):,}')
print(f'Pass-through stations (2 connections):  {len(pass_through):,}')
print(f'Junction stations     (3+ connections): {len(junctions):,}')
print()
print(f'Fraction that are simple pass-throughs: {len(pass_through)/len(connections)*100:.0f}%')

In [ ]:
# Map the dead ends
coords = stations.set_index('code')[['lat','lon']]

dead_codes = dead_ends.index.tolist()
dead_locs  = coords.loc[coords.index.isin(dead_codes)]
all_locs   = coords

fig, ax = plt.subplots(figsize=(8, 10))
ax.scatter(all_locs['lon'],  all_locs['lat'],  s=0.5, color='lightgrey', alpha=0.5, label='All stations')
ax.scatter(dead_locs['lon'], dead_locs['lat'], s=3,   color='tomato',    alpha=0.6, label=f'Dead ends ({len(dead_codes):,})')
ax.set_xlim(68, 97)
ax.set_ylim(8, 37)
ax.set_title('Dead end stations — one track connection only')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(markerscale=4)
plt.tight_layout()
plt.savefig('images/dead_ends.png', dpi=150)
plt.show()

**Write:** Where are dead end stations concentrated? What does this suggest about the strategy for expanding railways into remote or difficult terrain?

*Your answer:*

### Question 3 — How long is India's entire railway network?

In [ ]:
total_km = track['distance_km'].sum()

print(f'Total track length (straight-line): {total_km:,.0f} km')
print()
print('For comparison:')
print(f'  Circumference of Earth:  40,075 km')
print(f'  Our network is:          {total_km/40075*100:.0f}% of Earth circumference')
print()
print('Official Indian Railways figure: ~67,000 route km')
print(f'Our number ({total_km:,.0f} km) is lower because:')
print('  1. We measure straight-line distance between stations, not actual track curves')
print('  2. Some small branch lines may be missing from the schedule data')

### Question 4 — How far apart are stations from each other?

In [ ]:
avg_gap    = track['distance_km'].mean()
median_gap = track['distance_km'].median()

print(f'Average gap between neighboring stations: {avg_gap:.1f} km')
print(f'Median gap:                               {median_gap:.1f} km')
print()

# Which states have the densest and thinnest coverage?
by_state = track.groupby('state_a')['distance_km'].median()
by_state = by_state[track.groupby('state_a').size() >= 20]  # only states with enough data
by_state = by_state.sort_values()

print('States with densest rail coverage (smallest gap between stations):')
print(by_state.head(6).round(1).to_string())
print()
print('States with thinnest rail coverage (largest gap):')
print(by_state.tail(6).round(1).to_string())

### Question 5 — Which stretch of track carries the most trains?

In [ ]:
busiest = track.sort_values('num_trains', ascending=False)

print('15 busiest track connections (most trains per day):')
print()
print(f'{"From":<26} {"To":<26} {"Trains":>8} {"Distance":>10}')
print('-' * 72)
for _, row in busiest.head(15).iterrows():
    print(f"{str(row['name_a']):<26} {str(row['name_b']):<26} {int(row['num_trains']):>8} {row['distance_km']:>8.1f} km")

print()
print(f"Average distance for the 15 busiest connections: {busiest.head(15)['distance_km'].mean():.1f} km")
print('Observation: the busiest corridors are all very short — suburban commuter lines.')

**Write:** The busiest stretches of track are all very short — under 10 km. Why? What city are most of them near? What does this tell you about how most people actually use Indian Railways day to day?

*Your answer:*

---
## What we built today

From train timetables we:
- Understood what the raw schedule data contains
- Discovered that almost every Indian train stops at every station (genuine express trains are rare)
- Built the physical track network by removing express jumps using geometry
- Answered five real questions about India's railway system

**One question to think about before next session:**

If you had to shut down one station to cause the maximum disruption across the whole country — not the busiest station, not the most famous — which would you choose? And why might the answer be a station most people have never heard of?

**Next session:** Notebook 2 — Who Matters?